In [ ]:
# ==============================
# TITANIC SURVIVAL PREDICTION (ADVANCED)
# ==============================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Load Data
df = pd.read_csv("C:\Users\Ankith\Downloads\archive\Titanic-Dataset.csv")

# ==============================
# FEATURE ENGINEERING ⭐
# ==============================

# Title extraction
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# Group rare titles
rare_titles = ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona']
df['Title'] = df['Title'].replace(rare_titles, 'Rare')

# Family features
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# Age handling
df['Age'].fillna(df['Age'].median(), inplace=True)
df['AgeGroup'] = pd.cut(df['Age'], bins=[0,12,20,40,60,80], labels=[0,1,2,3,4])

# Fare handling
df['Fare'].fillna(df['Fare'].median(), inplace=True)
df['FareBand'] = pd.qcut(df['Fare'], 4, labels=[0,1,2,3])

# Embarked
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

# Drop unused
df.drop(['Name','Ticket','Cabin','PassengerId'], axis=1, inplace=True)

# ==============================
# ENCODING
# ==============================

le = LabelEncoder()
for col in ['Sex','Embarked','Title']:
    df[col] = le.fit_transform(df[col])

df['AgeGroup'] = df['AgeGroup'].astype(int)
df['FareBand'] = df['FareBand'].astype(int)

# ==============================
# SPLIT
# ==============================

X = df.drop('Survived', axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# ==============================
# MODEL + TUNING ⭐
# ==============================

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [4, 6, 8],
    'min_samples_split': [2, 5]
}

grid = GridSearchCV(RandomForestClassifier(), param_grid, cv=5)
grid.fit(X_train, y_train)

# ==============================
# EVALUATION
# ==============================

y_pred = grid.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

sns.heatmap(confusion_matrix(y_test, y_pred), annot=True)
plt.show()

# ==============================
# SAVE MODEL
# ==============================

pickle.dump(grid.best_estimator_, open("../model/model.pkl", "wb"))

print("Model saved!")